# Simple unary binary-string tasks

Train the same local NCA on reversal or bitwise NOT. Leading zeroes are data and are preserved. All experiment hyperparameters are defined below.

In [ ]:
from dataclasses import replace
from pathlib import Path
import sys

import torch
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "run":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ncpu_computer import (
    ExperimentConfig, GeometryConfig, ModelConfig, TapeLayout, TaskDataset,
    TrainingConfig, bitwise_not_task, encode_strings, evaluate, infer,
    load_model, reverse_task, save_gif, train_seeds, validate_experiment,
)
from ncpu_computer.evaluation import format_results

# Task and tape geometry
TASK_NAME = "reverse"  # "reverse" or "bit_not"
TRAIN_MAX_LENGTH = 6
EXTRAPOLATION_LENGTH = 8
TAIL_SLOTS = 0
STRIDE = 2
BORDER_LEFT = 3
BORDER_RIGHT = 3
BORDER_TOP = 3
BORDER_BOTTOM = 3

# Local rule
CHANNELS = 5
HIDDEN_SIZE = 64
FIXED_KERNELS = ("identity", "sobel_x", "sobel_y")
FIXED_LAPLACIAN = False
LEARNABLE_KERNELS = 0
LEARNABLE_KERNEL_INIT = "laplacian"
GATE = "sigmoid"
GATE_BIAS = 1.0
FIRE_RATE = 0.9
PROGRAM_CHANNEL = 0
IO_CHANNEL = 1
PADDING = "zeros"
MAX_ABS_STATE = 10.0
RANDOM_KERNEL_SEED = 0

# Training
UPDATES = 1400
BATCH_SIZE = 128
FREE_STEPS = 50
SUPERVISION_STEPS = 100
LEARNING_RATE = 5e-3
FINAL_LEARNING_RATE = 1e-4
WARMUP_UPDATES = 0
WEIGHT_DECAY = 2e-5
GRAD_CLIP = 0.8
TERMINATOR_WEIGHT = 0.0
TAIL_WEIGHT = 0.0
TRAINING_SEED = 0
VALIDATION_EVERY = 50
CHECKPOINT_EVERY = 50
DEVICE = "auto"
SEEDS = (0,)

# Notebook actions
RUN_TRAINING = False
RESUME = True
RUN_EVALUATION = False
RUN_INFERENCE = False
RUN_VISUALIZATION = False

# Evaluation, inference, and visualization
EVALUATION_BATCH_SIZE = 128
SAMPLE_INPUT = "111011"
INFERENCE_STEPS = 150
GIF_DURATION_MS = 150
GIF_SCALE = 32

task_builders = {"reverse": reverse_task, "bit_not": bitwise_not_task}
if TASK_NAME not in task_builders:
    raise ValueError(f"unknown task: {TASK_NAME}")
task_builder = task_builders[TASK_NAME]

geometry = GeometryConfig(
    tape_slots=TRAIN_MAX_LENGTH + 1 + TAIL_SLOTS,
    stride=STRIDE,
    border_left=BORDER_LEFT,
    border_right=BORDER_RIGHT,
    border_top=BORDER_TOP,
    border_bottom=BORDER_BOTTOM,
)
model_config = ModelConfig(
    channels=CHANNELS,
    hidden_size=HIDDEN_SIZE,
    fixed_kernels=FIXED_KERNELS,
    fixed_laplacian=FIXED_LAPLACIAN,
    learnable_kernels=LEARNABLE_KERNELS,
    learnable_kernel_init=LEARNABLE_KERNEL_INIT,
    gate=GATE,
    gate_bias=GATE_BIAS,
    fire_rate=FIRE_RATE,
    program_channel=PROGRAM_CHANNEL,
    io_channel=IO_CHANNEL,
    padding=PADDING,
    max_abs_state=MAX_ABS_STATE,
    random_kernel_seed=RANDOM_KERNEL_SEED,
)
training_config = TrainingConfig(
    updates=UPDATES,
    batch_size=BATCH_SIZE,
    free_steps=FREE_STEPS,
    supervision_steps=SUPERVISION_STEPS,
    learning_rate=LEARNING_RATE,
    final_learning_rate=FINAL_LEARNING_RATE,
    warmup_updates=WARMUP_UPDATES,
    weight_decay=WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    terminator_weight=TERMINATOR_WEIGHT,
    tail_weight=TAIL_WEIGHT,
    seed=TRAINING_SEED,
    validation_every=VALIDATION_EVERY,
    checkpoint_every=CHECKPOINT_EVERY,
    device=DEVICE,
)
config = ExperimentConfig(geometry, model_config, training_config)
train_task = task_builder(TRAIN_MAX_LENGTH, include_shorter=True)
train_data = TaskDataset.from_task(train_task, geometry.tape_slots)
layout = TapeLayout(geometry)
checkpoint_dir = ROOT / "checkpoints" / f"{TASK_NAME}_train{TRAIN_MAX_LENGTH}"
checkpoint_path = checkpoint_dir / "best.pt"
gif_path = ROOT / "run" / f"{TASK_NAME}_evolution.gif"

print(validate_experiment(config, train_data))
print("\nTape layout:")
print(layout.schema())
print(f"\ntask: {train_task.name}; examples: {len(train_data)}")
for example in train_task.examples[:8]:
    print(f"{example.input:>{TRAIN_MAX_LENGTH}} -> {example.target}")

In [ ]:
if RUN_TRAINING:
    seed_results = train_seeds(
        config, train_data, seeds=SEEDS, checkpoint_dir=checkpoint_dir,
        resume=RESUME,
    )
    seed_results

In [ ]:
if RUN_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    if checkpoint["dataset_signature"] != train_data.signature:
        raise ValueError("checkpoint dataset differs from this notebook")
    schedule = trained_config.training
    train_result = evaluate(
        model, trained_config.geometry, train_data,
        steps=schedule.rollout_steps,
        step_start=schedule.supervision_start,
        step_end=schedule.supervision_end,
        batch_size=EVALUATION_BATCH_SIZE,
    )
    extrapolation_geometry = replace(
        trained_config.geometry,
        tape_slots=EXTRAPOLATION_LENGTH + 1 + TAIL_SLOTS,
    )
    extrapolation_task = task_builder(
        EXTRAPOLATION_LENGTH, include_shorter=False
    )
    extrapolation_data = TaskDataset.from_task(
        extrapolation_task, extrapolation_geometry.tape_slots
    )
    extrapolation_result = evaluate(
        model, extrapolation_geometry, extrapolation_data,
        steps=schedule.rollout_steps,
        step_start=schedule.supervision_start,
        step_end=schedule.supervision_end,
        batch_size=EVALUATION_BATCH_SIZE,
    )
    print(format_results([train_result, extrapolation_result]))

In [ ]:
if RUN_INFERENCE:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    inference_geometry = replace(
        trained_config.geometry, tape_slots=len(SAMPLE_INPUT) + 1 + TAIL_SLOTS
    )
    sample_task = task_builder(len(SAMPLE_INPUT), include_shorter=False)
    expected = next(
        example.target for example in sample_task.examples
        if example.input == SAMPLE_INPUT
    )
    prediction = infer(
        model, inference_geometry, SAMPLE_INPUT, steps=INFERENCE_STEPS
    )
    predicted_string = (
        prediction.interpreted.binary_strings[0]
        if prediction.interpreted.valid else None
    )
    print(f"continuous tape: {prediction.values.tolist()}")
    print(f"raw ternary:    {prediction.interpreted.raw}")
    print(f"prediction:     {predicted_string}")
    print(f"expected:       {expected}")

In [ ]:
if RUN_VISUALIZATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, trained_config, checkpoint = load_model(checkpoint_path, DEVICE)
    visualization_geometry = replace(
        trained_config.geometry, tape_slots=len(SAMPLE_INPUT) + 1 + TAIL_SLOTS
    )
    visualization_layout = TapeLayout(visualization_geometry)
    sample_task = task_builder(len(SAMPLE_INPUT), include_shorter=False)
    expected = next(
        example.target for example in sample_task.examples
        if example.input == SAMPLE_INPUT
    )
    encoded = encode_strings(
        (SAMPLE_INPUT,), visualization_geometry.tape_slots
    ).to(model.device)
    initial = model.initial_state(visualization_layout.render_tape(encoded))
    with torch.inference_mode():
        rollout = model(initial, INFERENCE_STEPS)[0].cpu()
    saved_gif = save_gif(
        rollout, gif_path, layout=visualization_layout, config=trained_config,
        input_symbols=SAMPLE_INPUT, target_symbols=expected,
        output_mode=train_data.output_mode, duration_ms=GIF_DURATION_MS,
        scale=GIF_SCALE,
    )
    display(Image(filename=str(saved_gif)))
    print(saved_gif)